## **Phishing Website Detection Using Machine Learning**

In [1]:
# Import Data Manipulation Library
import pandas as pd
import numpy as np
# Import Data Visualization Library
import seaborn as sns
import matplotlib.pyplot as plt 
# Import Filter Warning Libraries
import warnings
warnings.filterwarnings(action = 'ignore')
# Import Logging Libraries
import logging 
logging.basicConfig(level = logging.INFO,
                    filename = 'model.log',
                    filemode = 'w',
                    format = "%(asctime)s - %(levelname)s - %(message)s",
                    force = True)
# Import Sci-kit Learn Libraries
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report,accuracy_score,confusion_matrix

# Import OrderedDict Library
from collections import OrderedDict

# Set theme style
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
# Import Machine Learning Models 

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier

In [2]:
def data_ingestion():
    df = pd.read_csv(r'C:\PhisingData_PredictionModel\data\raw\data.csv')
    return df


# Data Exploration

def data_exploration(df):

    stats = []
    numerical_cols = df.select_dtypes(exclude='object').columns

    for col in numerical_cols:

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1

        lower_whisker = Q1 - 1.5 * IQR
        upper_whisker = Q3 + 1.5 * IQR

        outlier_flag = (
            "Has Outliers"
            if df[(df[col] < lower_whisker) | (df[col] > upper_whisker)].shape[0] > 0
            else "No Outliers"
        )

        numerical_stats = OrderedDict({
            "Feature": col,
            "Minimum": df[col].min(),
            "Maximum": df[col].max(),
            "Mean": df[col].mean(),
            "Median": df[col].median(),
            "Mode": df[col].mode().iloc[0] if not df[col].mode().empty else np.nan,
            "25%": Q1,
            "75%": Q3,
            "IQR": IQR,
            "Standard Deviation": df[col].std(),
            "Skewness": df[col].skew(),
            "Kurtosis": df[col].kurt(),
            "Outlier Comment": outlier_flag,
            "Unique Values": df[col].nunique()
        })

        stats.append(numerical_stats)

    return pd.DataFrame(stats)


# Categorical Summary

def categorical_summary(df):

    cat_cols = df.select_dtypes(include='object').columns
    summary = []

    for col in cat_cols:
        summary.append({
            "Feature": col,
            "Unique Values": df[col].nunique(),
            "Most Frequent": df[col].mode().iloc[0] if not df[col].mode().empty else None,
            "Missing Values": df[col].isna().sum()
        })

    return pd.DataFrame(summary)


# Data Preprocessing

def data_preprocessing(df):

    df = df.copy()

    if 'url' in df.columns:
        df.drop(columns=['url'], inplace=True)

    target = 'status'

    le = LabelEncoder()
    df[target] = le.fit_transform(df[target])

    y = df[target]
    X = df.drop(columns=[target])

    constant_cols = [col for col in X.columns if X[col].nunique() == 1]
    X.drop(columns=constant_cols, inplace=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=1,
        stratify=y
    )

    numerical_cols = X_train.select_dtypes(exclude='object').columns
    categorical_cols = X_train.select_dtypes(include='object').columns

    num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ])

    cat_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer([
        ('num', num_pipeline, numerical_cols),
        ('cat', cat_pipeline, categorical_cols)
    ])

    final_pipeline = ImbPipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('smote', SMOTE(random_state=1))
    ])

    X_train_resampled, y_train_resampled = final_pipeline.fit_resample(X_train, y_train)

    X_test_transformed = final_pipeline.named_steps['preprocessor'].transform(X_test)
    X_test_transformed = final_pipeline.named_steps['pca'].transform(X_test_transformed)

    return X_train_resampled, X_test_transformed, y_train_resampled, y_test


# Model Building

def model_building(X_train, X_test, y_train, y_test):

    models = {
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=1),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=200, random_state=1),
        "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=1),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=200, random_state=1),
        "XGBoost": XGBClassifier(n_estimators=200, random_state=1, eval_metric='logloss')
    }

    results = []

    for name, model in models.items():

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        results.append({
            "Model": name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred),
            "F1 Score": f1_score(y_test, y_pred)
        })

    results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)

    logging.info("\n%s", results_df)

    return results_df


# Stratified KFold Validation

def kfold_validation(X, y, n_splits=5):

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=1)

    models = {
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=1),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=200, random_state=1),
        "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=1),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=200, random_state=1),
        "XGBoost": XGBClassifier(n_estimators=200, random_state=1, eval_metric='logloss')
    }

    results = []

    for model_name, model in models.items():

        f1_scores = []

        for train_index, test_index in skf.split(X, y):

            X_train_fold, X_test_fold = X[train_index], X[test_index]
            y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]

            model.fit(X_train_fold, y_train_fold)
            y_pred = model.predict(X_test_fold)

            f1_scores.append(f1_score(y_test_fold, y_pred))

        results.append({
            "Model": model_name,
            "Average F1 Score": np.mean(f1_scores)
        })

    return pd.DataFrame(results).sort_values(by="Average F1 Score", ascending=False)

In [3]:
def main():

    print("Phishing Detection Model Execution Started")

    # Data Ingestion
    df = data_ingestion()
    print("Data Ingestion Completed")

    # Data Exploration
    numerical_report = data_exploration(df)
    categorical_report = categorical_summary(df)

    print("Data Exploration Completed")
    print("Numerical Features Summary Shape:", numerical_report.shape)
    print("Categorical Features Summary Shape:", categorical_report.shape)

    # Data Preprocessing
    X_train, X_test, y_train, y_test = data_preprocessing(df)
    print("Data Preprocessing Completed")
    print("Training Data Shape:", X_train.shape)
    print("Testing Data Shape:", X_test.shape)

    # Model Building and Evaluation
    results_df = model_building(X_train, X_test, y_train, y_test)
    print("Model Training and Evaluation Completed")

    # KFold Cross Validation (Using Training Data Only)
    kfold_results = kfold_validation(X_train, y_train)
    print("KFold Cross Validation Completed")

    print("\nFinal Test Set Performance")
    print(results_df)

    print("\nKFold Cross Validation Results")
    print(kfold_results)

    print("Phishing Detection Model Execution Finished Successfully")


# Entry Point

main()

Phishing Detection Model Execution Started
Data Ingestion Completed
Data Exploration Completed
Numerical Features Summary Shape: (87, 14)
Categorical Features Summary Shape: (2, 4)
Data Preprocessing Completed
Training Data Shape: (8002, 29)
Testing Data Shape: (3429, 29)
Model Training and Evaluation Completed
KFold Cross Validation Completed

Final Test Set Performance
              Model  Accuracy  Precision    Recall  F1 Score
0      RandomForest  0.944882   0.946721  0.942857  0.944785
4           XGBoost  0.944590   0.948266  0.940525  0.944379
1        ExtraTrees  0.944882   0.954708  0.934111  0.944297
3  GradientBoosting  0.939633   0.941970  0.937026  0.939491
2          AdaBoost  0.926801   0.926077  0.927697  0.926886

KFold Cross Validation Results
              Model  Average F1 Score
4           XGBoost          0.942855
1        ExtraTrees          0.939131
0      RandomForest          0.935456
3  GradientBoosting          0.934935
2          AdaBoost          0.916565
